# Análisis de Administradoras de Crédito Uruguay — BCU

Fuente: BCU — Boletín SSF, Administradoras de Crédito > 150.000 UR  
Cifras en miles de pesos uruguayos.  

**Empresa foco**: configurada en `config/settings.py` (FOCUS_COMPANY_CODE / FOCUS_COMPANY_NAME)

In [ ]:
import sys
from pathlib import Path

# Asegurar que los módulos del proyecto estén disponibles
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from config.settings import (
    FOCUS_COMPANY_CODE, FOCUS_COMPANY_NAME, FOCUS_COMPANY_SHORT,
    FOCUS_COMPANY_COLOR, SECTOR_CODE,
)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

PEERS = {'804': 'SOCUR', '805': 'ANDA', '815': 'OCA SA', '860': 'FUCAC', FOCUS_COMPANY_CODE: FOCUS_COMPANY_SHORT}
PEER_PALETTE = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', FOCUS_COMPANY_COLOR]
MESES_ES = {1:'ene',2:'feb',3:'mar',4:'abr',5:'may',6:'jun',7:'jul',8:'ago',9:'set',10:'oct',11:'nov',12:'dic'}

In [ ]:
# Cargar datos consolidados
csv = Path('data/processed/serie_temporal.csv')
if not csv.exists():
    print('Primero correr: python parser.py')
else:
    df = pd.read_csv(csv, parse_dates=['fecha'])
    df['codigo'] = df['codigo'].astype(str)
    print(f'Dataset: {len(df)} filas | {df["fecha"].nunique()} meses | {df["codigo"].nunique()} instituciones')
    print(f'Período: {df["fecha"].min().strftime("%b %Y")} → {df["fecha"].max().strftime("%b %Y")}')
    df.head()

## 1. Snapshot empresa foco — Último mes disponible

In [ ]:
ultimo_mes = df['fecha'].max()
foco_ult = df[(df['codigo'] == FOCUS_COMPANY_CODE) & (df['fecha'] == ultimo_mes)].iloc[0]
sector_ult = df[(df['codigo'] == SECTOR_CODE) & (df['fecha'] == ultimo_mes)].iloc[0]

ms = foco_ult['cartera_bruta'] / sector_ult['cartera_bruta'] * 100
ratio_det = abs(foco_ult['deterioro_balance']) / foco_ult['cartera_bruta'] * 100

print(f'=== {FOCUS_COMPANY_NAME} — {ultimo_mes.strftime("%B %Y")} ===')
print(f'Market share cartera bruta : {ms:.1f}%')
print(f'Ratio de deterioro          : {ratio_det:.1f}%')
print(f'Resultado operativo         : ${foco_ult["resultado_operativo"]/1000:,.0f}M')
print(f'Cartera bruta               : ${foco_ult["cartera_bruta"]/1000:,.0f}M')
print(f'Activos totales             : ${foco_ult["activos_total"]/1000:,.0f}M')
print(f'Patrimonio                  : ${foco_ult["patrimonio"]/1000:,.0f}M')

## 2. Market Share — Evolución de cartera bruta

In [ ]:
sector = df[df['codigo'] == SECTOR_CODE][['fecha','cartera_bruta']].rename(columns={'cartera_bruta':'sector_total'})
merged = df[df['codigo'].isin(PEERS)].merge(sector, on='fecha')
merged['market_share'] = merged['cartera_bruta'] / merged['sector_total'] * 100

fig, ax = plt.subplots(figsize=(12, 5))
for i, (codigo, nombre) in enumerate(PEERS.items()):
    data = merged[merged['codigo'] == codigo].sort_values('fecha')
    if data.empty: continue
    color = FOCUS_COMPANY_COLOR if codigo == FOCUS_COMPANY_CODE else PEER_PALETTE[i]
    ax.plot(data['fecha'], data['market_share'], label=nombre,
            color=color, linewidth=2.5 if codigo==FOCUS_COMPANY_CODE else 1.5, marker='o', markersize=4)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.set_title('Market Share — Cartera Bruta', fontsize=13)
ax.set_ylabel('Share del sector (%)')
ax.legend()
fig.tight_layout()

## 3. Ratio de Deterioro

In [ ]:
peers_df = df[df['codigo'].isin(PEERS)].copy()
peers_df = peers_df[peers_df['cartera_bruta'].notna() & (peers_df['cartera_bruta'] > 0)]
peers_df['ratio_deterioro'] = peers_df['deterioro_balance'].abs() / peers_df['cartera_bruta'] * 100

fig, ax = plt.subplots(figsize=(12, 5))
for i, (codigo, nombre) in enumerate(PEERS.items()):
    data = peers_df[peers_df['codigo'] == codigo].sort_values('fecha')
    if data.empty: continue
    color = FOCUS_COMPANY_COLOR if codigo == FOCUS_COMPANY_CODE else PEER_PALETTE[i]
    ax.plot(data['fecha'], data['ratio_deterioro'], label=nombre,
            color=color, linewidth=2.5 if codigo==FOCUS_COMPANY_CODE else 1.5, marker='o', markersize=4)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.set_title('Ratio de Deterioro — Deterioro / Cartera Bruta', fontsize=13)
ax.set_ylabel('Ratio deterioro (%)')
ax.legend()
fig.tight_layout()

## 4. Resultado Operativo

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for i, (codigo, nombre) in enumerate(PEERS.items()):
    data = df[(df['codigo'] == codigo) & df['resultado_operativo'].notna()].sort_values('fecha')
    if data.empty: continue
    color = FOCUS_COMPANY_COLOR if codigo == FOCUS_COMPANY_CODE else PEER_PALETTE[i]
    ax.plot(data['fecha'], data['resultado_operativo']/1000, label=nombre,
            color=color, linewidth=2.5 if codigo==FOCUS_COMPANY_CODE else 1.5, marker='o', markersize=4)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.set_title('Resultado Operativo (millones de pesos)', fontsize=13)
ax.legend()
fig.tight_layout()

## 5. Comparativa peers — Último mes

In [ ]:
ult = df[(df['fecha'] == ultimo_mes) & df['codigo'].isin(PEERS)].copy()
ult = ult[ult['cartera_bruta'].notna() & (ult['cartera_bruta'] > 0)]
ult['ratio_deterioro'] = ult['deterioro_balance'].abs() / ult['cartera_bruta'] * 100
ult['rent_cartera'] = ult['margen_financiero_bruto'] / ult['cartera_bruta'] * 100
ult['nombre'] = ult['codigo'].map(PEERS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Deterioro
ult_d = ult.sort_values('ratio_deterioro')
colors = [FOCUS_COMPANY_COLOR if c==FOCUS_COMPANY_CODE else '#457B9D' for c in ult_d['codigo']]
axes[0].barh(ult_d['nombre'], ult_d['ratio_deterioro'], color=colors)
axes[0].xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
axes[0].set_title('Ratio de Deterioro')

# Rentabilidad
ult_r = ult.sort_values('rent_cartera')
colors2 = [FOCUS_COMPANY_COLOR if c==FOCUS_COMPANY_CODE else '#457B9D' for c in ult_r['codigo']]
axes[1].barh(ult_r['nombre'], ult_r['rent_cartera'], color=colors2)
axes[1].xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
axes[1].set_title('Margen Financiero / Cartera Bruta')

fig.suptitle(f'{FOCUS_COMPANY_NAME} vs Peers — {ultimo_mes.strftime("%B %Y")}', fontsize=13, fontweight='bold')
fig.tight_layout()

## 6. Resumen ejecutivo

In [ ]:
resumen_path = Path('output/resumen_ejecutivo.txt')
if resumen_path.exists():
    print(resumen_path.read_text(encoding='utf-8'))
else:
    print('Correr analysis.py primero para generar el resumen.')